In [ ]:
# @title Установка

import os
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/content/semantic-study-search")
APP_PYTHON = PROJECT / ".venv/bin/python"

if not Path("/content").is_dir():
    raise RuntimeError("Откройте этот блокнот в Google Colab.")
if globals().get("app") is not None and globals()["app"].poll() is None:
    raise RuntimeError("Приложение уже работает. Для перезапуска выполните ячейку «Запуск».")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv==0.11.29"],
    check=True,
)
if not PROJECT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/AynurAkhmadullin/semantic-study-search.git",
            str(PROJECT),
        ],
        check=True,
    )
elif not (PROJECT / "pyproject.toml").is_file():
    raise RuntimeError(f"В каталоге {PROJECT} нет проекта.")

os.chdir(PROJECT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "sync",
        "--locked",
        "--all-extras",
        "--no-dev",
        "--python",
        "3.13",
    ],
    cwd=PROJECT,
    check=True,
)

LIBRARY = Path("/content/sss-library")
LIBRARY.mkdir(parents=True, exist_ok=True)
print("Установка завершена. Выполните ячейку «Запуск».")

In [ ]:
# @title Запуск

import re
import socket
import time
from http.cookies import SimpleCookie
from urllib.parse import urlsplit

import requests
from google.colab import output

LOG_PATH = Path("/content/sss.log")


def configure_streamlit(config_path: Path) -> None:
    """Настраивает XSRF-cookie для интерфейса внутри Colab.

    Args:
        config_path: Конфигурация копии проекта в Colab.
    """
    config_text = config_path.read_text(encoding="utf-8")
    # Удаляем старые значения, чтобы повторный запуск не создавал дубликаты.
    for key in ("xsrfCookieSameSite", "enableXsrfProtection", "enableCORS"):
        config_text = re.sub(rf"(?m)^[ \t]*{key}[ \t]*=.*\n?", "", config_text)
    settings = (
        '[server]\nxsrfCookieSameSite = "none"\nenableXsrfProtection = true\nenableCORS = true'
    )
    if "[server]" in config_text:
        config_text = config_text.replace("[server]", settings, 1)
    else:
        config_text = settings + "\n\n" + config_text
    config_path.write_text(config_text, encoding="utf-8")


def stop_app() -> None:
    """Останавливает запущенный блокнотом процесс и его сервисы."""
    if "app" in globals() and app.poll() is None:
        app.terminate()
        app.wait(timeout=45)


def check_ports_free() -> None:
    """Проверяет, что API и интерфейс полностью остановлены."""
    for port in (8000, 8501):
        with socket.socket() as probe:
            try:
                probe.bind(("127.0.0.1", port))
            except OSError as error:
                raise RuntimeError(
                    f"Порт {port} занят. Проверьте предыдущий запуск приложения."
                ) from error


if not LIBRARY.is_dir() or not APP_PYTHON.is_file():
    raise RuntimeError("Сначала выполните ячейку «Установка».")

stop_app()
check_ports_free()
configure_streamlit(PROJECT / ".streamlit/config.toml")

proxy_url = output.eval_js("google.colab.kernel.proxyPort(8501)")
proxy = urlsplit(proxy_url)
if proxy.scheme != "https" or not proxy.hostname:
    raise RuntimeError("Colab не вернул HTTPS-адрес. Обновите страницу и повторите эту ячейку.")

app_env = os.environ.copy()
# Настройка берётся из файла даже после прежних экспериментов с окружением.
for variable in (
    "STREAMLIT_SERVER_XSRF_COOKIE_SAME_SITE",
    "STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION",
    "STREAMLIT_SERVER_ENABLE_CORS",
):
    app_env.pop(variable, None)
app_env.update(
    SSS_INDEX_DIR=str(LIBRARY),
    SSS_UPLOAD_DIR=str(LIBRARY / "uploads"),
    SSS_MODEL_CACHE="/content/sss-model-cache",
    SSS_OFFLINE="0",
    STREAMLIT_BROWSER_SERVER_ADDRESS=proxy.hostname,
    STREAMLIT_BROWSER_SERVER_PORT="443",
)

print("Запускаем приложение и загружаем модель…")
with LOG_PATH.open("w", encoding="utf-8") as log:
    app = subprocess.Popen(
        [str(APP_PYTHON), "-u", "-m", "semantic_study_search.launcher"],
        cwd=PROJECT,
        env=app_env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

try:
    with requests.Session() as session:
        session.trust_env = False
        deadline = time.monotonic() + 900
        next_report = time.monotonic() + 30
        while time.monotonic() < deadline:
            if app.poll() is not None:
                raise RuntimeError("Приложение завершилось при запуске.")
            try:
                response = session.get("http://127.0.0.1:8501/_stcore/health", timeout=2)
                response.raise_for_status()
                api_response = session.get("http://127.0.0.1:8000/health", timeout=2)
                api_response.raise_for_status()
            except requests.RequestException:
                if time.monotonic() >= next_report:
                    print("Ожидаем готовности. Загрузка модели может занять несколько минут…")
                    next_report = time.monotonic() + 30
                time.sleep(1)
            else:
                break
        else:
            raise RuntimeError("Истекло время ожидания запуска (15 минут).")

    cookies = SimpleCookie()
    cookies.load(response.headers.get("Set-Cookie", ""))
    cookie = cookies.get("_streamlit_xsrf")
    same_site = cookie["samesite"] if cookie else ""
    secure = bool(cookie["secure"]) if cookie else False
    if same_site.lower() != "none" or not secure:
        raise RuntimeError(
            "Настройка XSRF не применилась: "
            f"SameSite={same_site or 'cookie отсутствует'}, Secure={secure}. "
            "Повторите ячейку «Запуск»."
        )
except BaseException:
    stop_app()
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise

print("Приложение готово.")
output.no_vertical_scroll()
output.serve_kernel_port_as_iframe(8501, height=1700)